## 撥放YouTube視訊
!pip install opencv-python yt-dlp

In [ ]:

!yt-dlp --js-runtimes node --remote-components ejs:github --list-formats https://www.youtube.com/watch?v=ouq0uoVPdak

[youtube] Extracting URL: https://www.youtube.com/watch?v=ouq0uoVPdak
[youtube] ouq0uoVPdak: Downloading webpage
[youtube] ouq0uoVPdak: Downloading android vr player API JSON
[youtube] ouq0uoVPdak: Downloading web safari player API JSON
[youtube] ouq0uoVPdak: Downloading player 4e67f8a0-tv
[youtube] [jsc:node] Solving JS challenges using node
[youtube] [jsc:node] Downloading challenge solver lib script from  https://github.com/yt-dlp/ejs/releases/download/0.5.0/yt.solver.lib.min.js
[youtube] ouq0uoVPdak: Downloading m3u8 information
[info] Available formats for ouq0uoVPdak:
ID  EXT   RESOLUTION FPS CH |   FILESIZE    TBR PROTO | VCODEC           VBR ACODEC      ABR ASR MORE INFO
------------------------------------------------------------------------------------------------------------------
sb2 mhtml 48x27        0    |                   mhtml | images                                   storyboard
sb1 mhtml 80x45        0    |                   mhtml | images                           

In [3]:
try:
    from yt_dlp import YoutubeDL
except:
    !pip install yt-dlp
    from yt_dlp import YoutubeDL
try:
    import cv2
except:
    !pip install opencv-python
    import cv2

import time
import yt_dlp

url      = "https://www.youtube.com/watch?v=lLY46UGteJ0" #【玉山國家公園】熊鷹小夫妻－夫(右)妻(左-伊布)
ydl_opts = {'format':'best',
            'js-runtimes':['node'],
            'remote-components': 'ejs:github'}

with YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)
    video_url = info['url']
    
video    = cv2.VideoCapture(video_url)

if video is not None:
    fps    = int(video.get(cv2.CAP_PROP_FPS))
    width  = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print('Frame rate:',fps,'Frame width:',width,'Frame height:',height)
    
    frame_num = 0
    play_flag = 1
    total_frame=int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    def set_frame_number(x): #進度條
        global frame_num, play_flag 
        if play_flag == 0:#暫停才能調整進度條
            frame_num = x
            video.set(cv2.CAP_PROP_POS_FRAMES,frame_num)
        return
    
    def play(x):
        global play_flag
        play_flag = x
        return

    cv2.namedWindow('youtube')
    cv2.createTrackbar('frame no.','youtube',0,total_frame-1,set_frame_number)
    cv2.createTrackbar('play','youtube',0,1,play)
    cv2.setTrackbarPos('play','youtube',play_flag)
    
    while True:
        
        cv2.setTrackbarPos('frame no.','youtube',frame_num)
        
        t1 = time.perf_counter_ns()
        if play_flag:
            grabbed, frame = video.read()#讀取影片的下一幀，grabbed是布林值，表示是否成功讀取到幀，frame是讀取到的幀的圖像數據
            if grabbed:
                cv2.imshow('youtube',frame)
                frame_num += 1
            else:
                break
        t2 = 1000//fps - (time.perf_counter_ns()-t1)//1000000 #計算剩餘的等待時間，確保每秒顯示fps幀，//是整數除法
        k = cv2.waitKey(max(t2,1))
        if k == 27:#27表示esc鍵
            break
            
    cv2.destroyAllWindows()        
    video.release()
    
else:
    print('cannot open {}'.format(url))  

[youtube] Extracting URL: https://www.youtube.com/watch?v=lLY46UGteJ0
[youtube] lLY46UGteJ0: Downloading webpage


[youtube] lLY46UGteJ0: Downloading android vr player API JSON
Frame rate: 30 Frame width: 640 Frame height: 360


error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window_w32.cpp:2582: error: (-27:Null pointer) NULL window: 'youtube' in function 'cvSetTrackbarPos'


## 撥放YouTube直播

In [1]:
try:
    from yt_dlp import YoutubeDL
except:
    !pip install yt-dlp
    from yt_dlp import YoutubeDL
try:
    import cv2
except:
    !pip install opencv-python
    import cv2

import numpy as np

try:
    from PIL import Image, ImageDraw, ImageFont
except:
    !pip install pillow
    from PIL import Image, ImageDraw, ImageFont
import time
import concurrent.futures
import threading

def connect_video(url):
    global ydl
    try:
        info = ydl.extract_info(url[0], download=False)
        video_url =info['url']
        video    = cv2.VideoCapture(video_url)    
        if video is not None:
            fps    = int(video.get(cv2.CAP_PROP_FPS))
            width  = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
            print('Frame rate:',fps,'Frame width:',width,'Frame height:',height)
            return ((video,url[1]),fps)    
        else:
            return None
    except Exception as e:
        print(e)
        return None
        
def read_video(i):
    global cont,frames,videos,fps,locks
    dt = 1/fps
    while cont:
        t1=time.time()
        grabbed, frame = videos[i][0].read()
        locks[i].acquire()
        frames[i] = frame
        locks[i].release()
        t2=time.time()
        delay_time = dt-(t2-t1)
        if delay_time>0:
            time.sleep(delay_time)
    

        
def cv2ImgAddText(img, text, pos, textColor=(0, 255, 0), textSize=20):
    if (isinstance(img, np.ndarray)):  #判斷是否OpenCV圖片類型
        img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img)
    fontText = ImageFont.truetype("C:\\Windows\\Fonts\\simsun.ttc",textSize,encoding="utf-8")
    draw.text(pos, text, textColor, font=fontText)
    return cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2BGR)

urls = [("https://www.youtube.com/watch?v=-tQr28-BR_0","碧砂漁港"), 
        ("https://www.youtube.com/watch?v=XSD5ptYisw8","九份"), 
        ("https://www.youtube.com/watch?v=Ndo_8RuefH4","大稻埕碼頭"),
        ("https://www.youtube.com/watch?v=z_fY1pj1VBw","象山看台北"),
        ("https://www.youtube.com/watch?v=fP4ecxfsJos","台北大佳河濱公園"),
        ("https://www.youtube.com/watch?v=y_eOttu7aHI","野柳"), 
       ("https://www.youtube.com/watch?v=GUCaVR88ZFU",'石門水庫'),
       ("https://www.youtube.com/watch?v=VgmWVBUsfGo",'雪霸國家公園-汶水遊客中心'),
        ("https://www.youtube.com/watch?v=YkIUZjVlhv4","拉拉山遊客中心"),
        ("https://www.youtube.com/watch?v=fjhg3gAnMFg","台中高美濕地"),
       ("https://www.youtube.com/watch?v=UCG1aXVO8H8","台東多良"),
       ("https://www.youtube.com/watch?v=ka7FV0sCvxQ","高雄旗津海水浴場"), #
       ("https://www.youtube.com/watch?v=dQ7Sd6PGLdA","台東三仙台"), 
       ("https://www.youtube.com/live/j2L_559nCjc?si=iqIHTLWimJh8IDxJ","阿里山"),
       ("https://www.youtube.com/live/RaTbGYKMUtk?si=8EvuMcJqdp3NwXNp","八卦山"),
       ("https://www.youtube.com/watch?v=xwAWSh35uuw","淡水漁人碼頭"),
      ] 

videos   = []
fps      = 1000
ydl_opts = {'format': 'best[ext=mp4]'}
ydl      = YoutubeDL(ydl_opts)

with concurrent.futures.ThreadPoolExecutor(max_workers=len(urls)) as executor:
    futures = [executor.submit(connect_video,url) for url in urls]
    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        if result:
            fps = min(fps,result[1])
            videos.append(result[0])
        
        
print(len(videos))

if videos:
  cv2.namedWindow('youtube live')
  big_frame = np.zeros(((len(videos)+3)//4*180,(4 if len(videos)>=4 else len(videos))*320,3),dtype=np.uint8)
  cont   = True
  frames = np.empty((len(videos),),dtype=object)
  locks = []  
  for i in range(len(videos)):
      locks.append(threading.Lock())
      
  threads= []
  for i in range(len(videos)):
      t = threading.Thread(target=read_video,args=(i,)) 
      threads.append(t)
      t.start()
    
  while True:
    t1 = time.time()
    for i in range(len(frames)):
        locks[i].acquire(False)
        f = frames[i]
        locks[i].release()
        if f is not None:
            frame = cv2.resize(f,(320,180))
            frame = cv2ImgAddText(frame,videos[i][1],(8,frame.shape[0]-24),(255, 255, 255))  
            row_idx = i//4
            col_idx = i%4
            big_frame[row_idx*frame.shape[0]:(row_idx+1)*frame.shape[0],
                    col_idx*frame.shape[1]:(col_idx+1)*frame.shape[1],:] = frame[:,:,:]  
    cv2.imshow('youtube live',big_frame)
    t2 = time.time()
    delay_time = 1000//fps-int((t2 - t1)*1000)  
    k = cv2.waitKey(delay_time if delay_time>0 else 5)
    if k == 27:
        cont = False
        break

      
  for thread in threads:
    thread.join()

  for lock in locks:
      del lock
    
  cv2.destroyAllWindows()        
  for video in videos:
    video[0].release()

    


[youtube] Extracting URL: https://www.youtube.com/watch?v=-tQr28-BR_0
[youtube] Extracting URL: https://www.youtube.com/watch?v=XSD5ptYisw8
[youtube] Extracting URL: https://www.youtube.com/watch?v=Ndo_8RuefH4
[youtube] Extracting URL: https://www.youtube.com/watch?v=z_fY1pj1VBw
[youtube] XSD5ptYisw8: Downloading webpage
[youtube] Ndo_8RuefH4: Downloading webpage
[youtube] Extracting URL: https://www.youtube.com/watch?v=fP4ecxfsJos
[youtube] Extracting URL: https://www.youtube.com/watch?v=y_eOttu7aHI
[youtube] Extracting URL: https://www.youtube.com/watch?v=GUCaVR88ZFU
[youtube] Extracting URL: https://www.youtube.com/watch?v=VgmWVBUsfGo
[youtube] -tQr28-BR_0: Downloading webpage
[youtube] Extracting URL: https://www.youtube.com/watch?v=YkIUZjVlhv4
[youtube] Extracting URL: https://www.youtube.com/watch?v=fjhg3gAnMFg
[youtube] z_fY1pj1VBw: Downloading webpage
[youtube] y_eOttu7aHI: Downloading webpage
[youtube] Extracting URL: https://www.youtube.com/watch?v=UCG1aXVO8H8
[youtube] Extra

[youtube] y_eOttu7aHI: Downloading android vr player API JSON


[youtube] -tQr28-BR_0: Downloading android vr player API JSON


[youtube] GUCaVR88ZFU: Downloading android vr player API JSON


[youtube] YkIUZjVlhv4: Downloading android vr player API JSON


...

[youtube] z_fY1pj1VBw: Downloading m3u8 information
[youtube] RaTbGYKMUtk: Downloading m3u8 information
[youtube] UCG1aXVO8H8: Downloading m3u8 information
[youtube] ka7FV0sCvxQ: Downloading android vr player API JSON


[youtube] XSD5ptYisw8: Downloading android vr player API JSON
[youtube] j2L_559nCjc: Downloading m3u8 information


[youtube] Ndo_8RuefH4: Downloading android vr player API JSON
[youtube] fP4ecxfsJos: Downloading android vr player API JSON
[youtube] dQ7Sd6PGLdA: Downloading android vr player API JSON


[youtube] xwAWSh35uuw: Downloading android vr player API JSON
[youtube] fjhg3gAnMFg: Downloading android vr player API JSON
[youtube] Ndo_8RuefH4: Downloading m3u8 information
[youtube] ka7FV0sCvxQ: Downloading m3u8 information
[youtube] XSD5ptYisw8: Downloading m3u8 information
[youtube] dQ7Sd6PGLdA: Downloading m3u8 information
[youtube] fP4ecxfsJos: Downloading m3u8 information
[youtube] fjhg3gAnMFg: Downloading m3u8 information
[youtube] xwAWSh35uuw: Downloading m3u8 information
Frame rate: 30 Frame width: 640 Frame height: 360
Frame rate: 30 Frame width: 1920 Frame height: 1080
Frame rate: 30 Frame width: 1920 Frame height: 1080
Frame rate: 30 Frame width: 1920 Frame height: 1080
Frame rate: 30 Frame width: 1920 Frame height: 1080
Frame rate: 30 Frame width: 1920 Frame height: 1080
Frame rate: 30 Frame width: 1920 Frame height: 1080
Frame rate: 30 Frame width: 1920 Frame height: 1080
Frame rate: 30 Frame width: 1920 Frame height: 1080
Frame rate: 30 Frame width: 1920 Frame height:

KeyboardInterrupt: 